# Extract Trade and OHLCV Features (Amberdata)

This script extracts the following fields:
- taker_imbalance
- signed_volume
- realized_vol_5m
- realized_vol_15m
- momentum_5m
- momentum_15m

An extract is created per Cryptocurrency coin:
- LTCUSDT
- BTCUSDT
- ETHUSDT
- SOLUSDT
- XRPUSDT
- DOGEUSDT
- BNBUSDT
- ADAUSDT
- LINKUSDT
- AVAXUSDT
- DOTUSDT
- BCHUSDT

In [1]:
import os
import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, UTC

# CONFIG
API_KEY = "" # insert key here
TRADES_URL = "https://api.amberdata.com/markets/futures/trades"
OHLCV_URL = "https://api.amberdata.com/markets/futures/ohlcv"
exchange = "binance"
instruments = ["LTCUSDT",
               "BTCUSDT",
               "ETHUSDT",
               "SOLUSDT",
               "XRPUSDT",
               "DOGEUSDT",
               "BNBUSDT",
               "ADAUSDT",
               "LINKUSDT",
               "AVAXUSDT",
               "DOTUSDT",
               "BCHUSDT",]

headers = {"x-api-key": API_KEY,"Accept": "application/json","Accept-Encoding": "gzip, deflate, br"}
BACKFILL_START_DATE = datetime(2025, 7, 1, tzinfo=UTC)
END_DATE = datetime.now(UTC)
CHUNK_DAYS = 7


# Functions
def get_symbol_name(instrument):
    return instrument.replace("USDT", "").lower()

def get_output_files(instrument):
    symbol = get_symbol_name(instrument)
    parquet_file = f"{symbol}_microstructure.parquet"
    csv_file = f"{symbol}_microstructure.csv"
    return parquet_file, csv_file

def load_existing_data(parquet_file):
    if not os.path.exists(parquet_file):
        return pd.DataFrame()
    existing_df = pd.read_parquet(parquet_file)
    if existing_df.empty:
        return pd.DataFrame()
    existing_df["timestamp"] = pd.to_datetime(existing_df["timestamp"],utc=True)
    existing_df = (existing_df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True))
    return existing_df

def determine_start_date(existing_df):
    if existing_df.empty:
        return BACKFILL_START_DATE
    last_timestamp = existing_df["timestamp"].max()
    return last_timestamp


def fetch_ohlcv(instrument, current_start, current_end):
    ohlcv_url = f"{OHLCV_URL}/{instrument}"
    ohlcv_params = {"exchange": exchange,"startDate": current_start.strftime("%Y-%m-%dT%H:%M:%SZ"),"endDate": current_end.strftime("%Y-%m-%dT%H:%M:%SZ"),"timeInterval": "minutes",timeFormat": "iso8601"}
    response = requests.get(ohlcv_url,headers=headers,params=ohlcv_params,timeout=60)
    response.raise_for_status()
    ohlcv_rows = (response.json().get("payload", {}).get("data", []))
    ohlcv_df = pd.DataFrame(ohlcv_rows)

    if ohlcv_df.empty:
        return pd.DataFrame()

    ohlcv_df["timestamp"] = pd.to_datetime(ohlcv_df["exchangeTimestamp"],utc=True)
    ohlcv_df = ohlcv_df.rename(columns={"volume": "ohlcv_volume"})
    numeric_columns = ["open","high","low","close","ohlcv_volume"]

    for col in numeric_columns:
        if col in ohlcv_df.columns:
            ohlcv_df[col] = pd.to_numeric(ohlcv_df[col],errors="coerce")

    ohlcv_df["log_return"] = np.log(ohlcv_df["close"] / ohlcv_df["close"].shift(1))
    ohlcv_df["momentum_5m"] = ohlcv_df["close"].pct_change(5)
    ohlcv_df["momentum_15m"] = ohlcv_df["close"].pct_change(15)
    ohlcv_df["realized_vol_5m"] = (ohlcv_df["log_return"].rolling(5).std())
    ohlcv_df["realized_vol_15m"] = (ohlcv_df["log_return"].rolling(15).std())
    return ohlcv_df


def fetch_trades(instrument, current_start, current_end):
    trades_url = f"{TRADES_URL}/{instrument}"
    trades_params = {"exchange": exchange,"startDate": current_start.strftime("%Y-%m-%dT%H:%M:%SZ"),"endDate": current_end.strftime("%Y-%m-%dT%H:%M:%SZ"),"timeFormat": "iso8601"}
    response = requests.get(trades_url,headers=headers,params=trades_params,timeout=60)
    response.raise_for_status()
    trades_rows = (response.json().get("payload", {}).get("data", []))
    trades_df = pd.DataFrame(trades_rows)

    if trades_df.empty:
        return pd.DataFrame()

    trades_df["timestamp"] = pd.to_datetime(trades_df["exchangeTimestamp"],utc=True)
    trades_df["minute"] = trades_df["timestamp"].dt.floor("min")
    trades_df["volume"] = pd.to_numeric(trades_df["volume"],errors="coerce")
    trades_df["isBuySide"] = trades_df["isBuySide"].astype(bool)
    trades_df["direction"] = np.where(trades_df["isBuySide"],1,-1)
    trades_df["signed_volume"] = (trades_df["volume"] * trades_df["direction"])
    trades_df["buy_volume"] = np.where(trades_df["isBuySide"],trades_df["volume"],0)
    trades_df["sell_volume"] = np.where(~trades_df["isBuySide"],trades_df["volume"],0)
    micro_df = (trades_df.groupby("minute").agg(buy_volume=("buy_volume", "sum"),sell_volume=("sell_volume", "sum"),signed_volume=("signed_volume", "sum")).reset_index())
    denominator = (micro_df["buy_volume"] + micro_df["sell_volume"])
    micro_df["taker_imbalance"] = np.where(denominator != 0,(micro_df["buy_volume"] - micro_df["sell_volume"]) / denominator,np.nan)
    return micro_df

def build_microstructure_chunk(instrument, current_start, current_end):
    ohlcv_df = fetch_ohlcv(instrument,current_start,current_end)
    if ohlcv_df.empty:
        print("No OHLCV rows found.")
        return pd.DataFrame()

    micro_df = fetch_trades(instrument,current_start,current_end)

    if micro_df.empty:
        print("No trade rows found.")
        return pd.DataFrame()

    merged = pd.merge(ohlcv_df,micro_df,left_on="timestamp",right_on="minute",how="left")
    merged = merged[["timestamp",
                     "open",
                     "high",
                     "low",
                     "close",
                     "ohlcv_volume",
                     "momentum_5m",
                     "momentum_15m",
                     "realized_vol_5m",
                     "realized_vol_15m",
                     "taker_imbalance",
                     "signed_volume"]]

    return merged

def extract_microstructure(instrument):
    parquet_file, csv_file = get_output_files(instrument)

    print("\n######################################################################")
    print(f"Starting {instrument}")
    print("######################################################################")

    existing_df = load_existing_data(parquet_file)
    current_start = determine_start_date(existing_df)
    print(f"Existing rows: {len(existing_df):,}")
    
    if not existing_df.empty:
        print(f"Existing latest timestamp: {existing_df['timestamp'].max()}")
    print(f"Incremental start: {current_start}")
    print(f"End date: {END_DATE}")

    if current_start >= END_DATE:
        print(f"{instrument} is already up to date.")
        return

    all_new_chunks = []

    while current_start < END_DATE:
        current_end = min(current_start + timedelta(days=CHUNK_DAYS),END_DATE)

        print(f"\nChunk: {current_start} → {current_end}")
        try:
            chunk_df = build_microstructure_chunk(instrument,current_start,current_end)

            if not chunk_df.empty:
                all_new_chunks.append(chunk_df)
                print(f"Fetched rows: {len(chunk_df):,}")
            current_start = current_end

        except Exception as e:
            print(f"Error in chunk {current_start} → {current_end}: {e}")
            current_start = current_end

    if all_new_chunks:
        new_df = pd.concat(all_new_chunks,ignore_index=True)
        print(f"\nNew rows before merge: {len(new_df):,}")
        final_df = pd.concat([existing_df, new_df],ignore_index=True)
    else:
        print("\nNo new rows fetched.")
        final_df = existing_df.copy()

    if final_df.empty:
        print(f"No final data for {instrument}. Skipping save.")
        return

    final_df["timestamp"] = pd.to_datetime(final_df["timestamp"],utc=True)
    final_df = (final_df.drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
    print(f"\nFinal shape {instrument}: {final_df.shape}")
    print(f"Final earliest timestamp: {final_df['timestamp'].min()}")
    print(f"Final latest timestamp: {final_df['timestamp'].max()}")
    final_df.to_parquet(parquet_file,index=False)
    final_df.to_csv(csv_file,index=False)
    print(f"Saved parquet: {parquet_file}")
    print(f"Saved csv: {csv_file}")
    print(f"Finished {instrument}")

# Run script
for instrument in instruments:
    extract_microstructure(instrument)

print("\nMicrostructure extraction complete.")


######################################################################
Starting LTCUSDT
######################################################################
Existing rows: 461,668
Existing latest timestamp: 2026-06-03 17:39:00+00:00
Incremental start: 2026-06-03 17:34:00+00:00
End date: 2026-06-12 20:13:31.907440+00:00

Chunk: 2026-06-03 17:34:00+00:00 → 2026-06-10 17:34:00+00:00
Fetched rows: 2,500

Chunk: 2026-06-10 17:34:00+00:00 → 2026-06-12 20:13:31.907440+00:00
Fetched rows: 3,039

New rows before merge: 5,539

Final shape LTCUSDT: (467201, 12)
Final earliest timestamp: 2025-07-01 00:00:00+00:00
Final latest timestamp: 2026-06-12 20:12:00+00:00
Saved parquet: ltc_microstructure.parquet
Saved csv: ltc_microstructure.csv
Finished LTCUSDT

######################################################################
Starting BTCUSDT
######################################################################
Existing rows: 461,675
Existing latest timestamp: 2026-06-03 17:39:00+00:00
Increment